# Classificador Fiscal Inteligente

**Automação de cadastro e classificação contábil com IA**

Classifica automaticamente itens e serviços dentro da taxonomia contábil da **sua empresa**,
sugerindo grupo, subgrupo, classe e conta contábil a partir de uma descrição textual.

Roda 100% no Google Colab, usando modelos de linguagem abertos via API gratuita (Groq).

---

## Como usar

Execute os blocos **na ordem**, de cima para baixo. Cada bloco tem uma explicação
seguida do código. Para rodar uma célula: clique nela e pressione **Shift + Enter**.

| Bloco | O que faz | Obrigatório? |
|-------|-----------|--------------|
| 1 | Instala as bibliotecas | Sim |
| 2 | Conecta ao modelo de IA (Groq) | Sim |
| 3 | Carrega SUA taxonomia | Sim |
| 4 | Carrega SEU plano de contas | Sim |
| 5 | Vincula e valida a integridade | Sim |
| 6 | Prepara a busca semântica | Sim |
| 7 | Carrega as funções de classificação | Sim |
| 7.1 | Classifica um item | Sim |
| 8 | Classifica uma planilha inteira | Opcional |
| 9 | Exporta os resultados | Opcional |
| 10 | Mede a precisão (validação) | Opcional |

---

### O que você precisa ter em mãos

1. **Uma conta gratuita no [Groq](https://console.groq.com)** para a API key
2. **Sua taxonomia** em JSON (estrutura de grupos, subgrupos e classes)
3. **Seu plano de contas** em CSV ou Excel (código e descrição das contas)

> Modelos de exemplo desses arquivos estão no repositório, na pasta `exemplos/`.


## Bloco 1 — Instalação das bibliotecas

Instala tudo que o notebook precisa. Roda uma vez só, demora cerca de 1 minuto.

> Se aparecerem avisos em vermelho sobre conflito de versões, pode ignorar —
> não afetam o funcionamento.

In [ ]:
!pip install -q groq chromadb sentence-transformers pandas openpyxl

print("\n✓ Bibliotecas instaladas")

## Bloco 2 — Conexão com o modelo de IA

Usamos a **API do Groq**, que roda modelos abertos de forma gratuita e muito rápida.

**Como obter sua API key:**
1. Acesse [console.groq.com](https://console.groq.com)
2. Crie uma conta gratuita
3. Vá em **API Keys** -> **Create API Key**
4. Copie a chave (começa com `gsk_...`)

---

### Recomendado: salvar a chave no cofre do Colab (configura uma vez só)

Assim você não digita a chave toda vez que abrir o notebook:

1. No menu lateral esquerdo, clique no ícone de **chave** (Secrets)
2. Clique em **Add new secret**
3. Em **Name**, escreva exatamente: `GROQ_API_KEY`
4. Em **Value**, cole sua chave
5. Ative o acesso ao notebook

Se configurar o Secret, o bloco lê a chave sozinho. Senão, ele pede manualmente.

In [ ]:
import os
from groq import Groq

api_key = None

# Tenta ler do cofre de Secrets do Colab
try:
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
    print("✓ Chave lida do cofre de Secrets do Colab")
except Exception:
    pass

# Se nao achou, pede manualmente
if not api_key:
    from getpass import getpass
    api_key = getpass("Cole sua GROQ API key e pressione Enter: ")

os.environ["GROQ_API_KEY"] = api_key
client = Groq(api_key=api_key)

# Modelo aberto, gratuito e rapido
MODELO = "openai/gpt-oss-120b"

# Teste de conexao
try:
    resp = client.chat.completions.create(
        model=MODELO,
        messages=[{"role": "user", "content": "Responda apenas: OK"}],
        max_tokens=100
    )
    print(f"✓ Conectado ao Groq — modelo {MODELO}")
except Exception as e:
    print(f"✗ Erro na conexao: {e}")

## Bloco 3 — Carregar SUA taxonomia

Faça upload do arquivo **taxonomia.json** da sua empresa.

A estrutura esperada é: **Tipo -> Grupo -> Subgrupo -> Classe -> conta_ref** (código reduzido da conta)

Cada Tipo tem o campo **natureza** (`Produto` ou `Servico`). No bloco 7 você informa se o item
é produto ou serviço e o modelo só enxerga os tipos daquela natureza. Isso evita confundir, por
exemplo, uma *máquina* (produto imobilizado) com a *manutenção da máquina* (serviço).

O modelo de exemplo traz três tipos: **1. Imobilizado** e **2. Uso e Consumo** (produtos) e
**9. Servicos**. Tipos particulares de cada empresa, como matéria-prima, insumos e produto acabado,
devem ser incluídos por quem adotar a automação, seguindo a mesma estrutura.

> Veja o modelo completo em `exemplos/taxonomia.json` no repositório.

In [ ]:
import json
from google.colab import files

print("Selecione o arquivo taxonomia.json da sua empresa:")
uploaded = files.upload()
nome = list(uploaded.keys())[0]

with open(nome, encoding="utf-8") as f:
    TAXONOMIA = json.load(f)

# Compatibilidade: se vier sem "tipos", assume um tipo unico
if "tipos" not in TAXONOMIA:
    if "grupos" in TAXONOMIA:
        TAXONOMIA = {
            "regras_imobilizacao": TAXONOMIA.get("regras_imobilizacao", {}),
            "tipos": {"Geral": {"grupos": TAXONOMIA["grupos"]}}
        }

total_tipos = len(TAXONOMIA["tipos"])
total_grupos = sum(len(t.get("grupos", {})) for t in TAXONOMIA["tipos"].values())
total_sub = sum(
    len(s.get("subgrupos", {}))
    for t in TAXONOMIA["tipos"].values()
    for s in t.get("grupos", {}).values()
)

print(f"\n✓ Taxonomia carregada")
print(f"  Tipos:     {total_tipos}")
print(f"  Grupos:    {total_grupos}")
print(f"  Subgrupos: {total_sub}")
print("\nEstrutura:")
for tipo, tdata in TAXONOMIA["tipos"].items():
    nat = tdata.get("natureza", "")
    print(f"  TIPO: {tipo}" + (f"  [{nat}]" if nat else ""))
    for g in tdata.get("grupos", {}):
        print(f"    - {g}")

## Bloco 4 — Carregar SEU plano de contas

Faça upload do plano de contas em **CSV ou Excel**.

O arquivo deve ter pelo menos duas colunas: **codigo** e **descricao**.

O modelo de exemplo segue a estrutura da Lei 6.404/76 (Ativo, Passivo, Receitas, Custos e Despesas)
e traz as colunas **mascara** (classificação, ex.: `4.2.2.02.001`), **codigo** (código reduzido),
**descricao** e **tipo_conta** (S = sintética, A = analítica). A taxonomia sempre aponta para o
código reduzido de uma conta analítica.

> Veja o modelo em `exemplos/plano_contas.csv` no repositório.

In [ ]:
import pandas as pd
from google.colab import files

print("Selecione o plano de contas (CSV ou Excel):")
uploaded = files.upload()
nome = list(uploaded.keys())[0]

if nome.endswith(".csv"):
    plano = pd.read_csv(nome, dtype=str)
else:
    plano = pd.read_excel(nome, dtype=str)

print(f"\n✓ Plano de contas carregado: {len(plano)} contas")
print("\nColunas encontradas:", list(plano.columns))
print("\nPrimeiras linhas:")
display(plano.head())

## Bloco 5 — Vincular e validar a integridade

Conecta a taxonomia ao plano de contas e verifica se **toda conta referenciada
na taxonomia realmente existe no plano**.

Cada informação tem um único dono: a taxonomia guarda a *referência* (código),
o plano de contas guarda os *detalhes* (descrição). Se houver referência quebrada,
o relatório avisa.

In [ ]:
def montar_indice_plano(plano_df):
    cols = {c.lower().strip(): c for c in plano_df.columns}
    col_codigo = col_desc = None
    for chave, original in cols.items():
        if chave in ("codigo", "conta", "cod"):
            col_codigo = original
        if chave in ("descricao", "titulo", "nome"):
            col_desc = original
    if not col_codigo or not col_desc:
        print("⚠ Colunas nao identificadas. Encontradas:", list(plano_df.columns))
        return {}
    indice = {}
    for _, row in plano_df.iterrows():
        cod = str(row[col_codigo]).strip()
        desc = str(row[col_desc]).strip()
        indice[cod] = desc
        indice[cod.lstrip("0") or cod] = desc
    return indice


def resolver_conta(ref, indice):
    ref = str(ref).strip()
    return indice.get(ref) or indice.get(ref.lstrip("0") or ref) or "(nao encontrada)"


INDICE_PLANO = montar_indice_plano(plano)
print(f"✓ Indice: {len(plano)} contas indexadas\n")

validas, orfas = 0, []
for tipo, tdata in TAXONOMIA["tipos"].items():
    for grupo, gdata in tdata.get("grupos", {}).items():
        for sub, sdata in gdata.get("subgrupos", {}).items():
            for classe, cdata in sdata.get("classes", {}).items():
                ref = str(cdata.get("conta_ref", "")).strip()
                if not ref:
                    continue
                if ref in INDICE_PLANO or (ref.lstrip("0") or ref) in INDICE_PLANO:
                    validas += 1
                else:
                    orfas.append({"tipo": tipo, "grupo": grupo, "subgrupo": sub, "classe": classe, "ref": ref})

print("="*55)
print("RELATORIO DE INTEGRIDADE")
print(f"  Referencias validas: {validas}")
print(f"  Referencias orfas:   {len(orfas)}")
print("="*55)
if orfas:
    print("\n⚠ Contas referenciadas na taxonomia que NAO existem no plano:\n")
    for o in orfas[:15]:
        print(f"  - {o['tipo']} > {o['grupo']} > {o['subgrupo']} -> conta {o['ref']}")
    if len(orfas) > 15:
        print(f"  ... e mais {len(orfas)-15}")
else:
    print("\n✓ Perfeito! Todas as referencias existem no plano de contas.")

## Bloco 6 — Preparar a busca semântica

Carrega o modelo de embeddings e cria a base vetorial (ChromaDB) em memória.

A base começa **vazia** e cresce conforme você classifica e valida itens —
sem catálogos externos que poluem a busca. Cada empresa constrói a própria base limpa.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

print("Carregando modelo de embeddings (primeira vez demora ~30s)...")
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

chroma = chromadb.Client()
try:
    chroma.delete_collection("itens")
except Exception:
    pass
colecao = chroma.create_collection("itens", metadata={"hnsw:space": "cosine"})

def gerar_embedding(texto):
    return embedder.encode(texto).tolist()

print("✓ Busca semantica pronta (base comeca vazia)")

## Bloco 6.2 — Carregar tabelas fiscais: NCM e Código de Serviço (OPCIONAL)

> Roda depois do Bloco 6, porque usa o modelo de embeddings e a base vetorial criados lá.

Para completar os campos fiscais do cadastro, o sistema pode sugerir:
- **NCM** (Nomenclatura Comum do Mercosul) — para **produtos**
- **Código de Serviço** (Lista da LC 116/2003) — para **serviços**, usado na retenção de ISS

A tabela NCM traz muitas descrições que só fazem sentido junto do nível superior, como
"Outros" ou "De carga radial". Por isso o arquivo de exemplo tem o campo **descricao_busca**,
que junta a descrição do item à posição e à subposição a que ele pertence. A busca usa esse
campo; o que aparece para você continua sendo a descrição oficial.

> **Opcional, mas recomendado.** Faça upload dos arquivos `ncm.json` e `codigo_servico.json`
> (ambos na pasta `exemplos/` do repositório). Você pode subir um, outro, ou os dois.

In [ ]:
import json
from google.colab import files

NCM_LISTA, NCM_INDEXADO = [], False
SERVICO_LISTA, SERVICO_INDEXADO = [], False

print("Anexe ncm.json e/ou codigo_servico.json (pode selecionar os dois):")
try:
    uploaded = files.upload()
    for nome in uploaded.keys():
        with open(nome, encoding="utf-8") as f:
            dados = json.load(f)

        # Detecta se e NCM ou codigo de servico
        if "ncms" in dados or (isinstance(dados, dict) and "ncm" in nome.lower()):
            NCM_LISTA = dados.get("ncms", [])
            print(f"\n✓ NCM: {len(NCM_LISTA)} codigos. Indexando (~1 min)...")
            try:
                chroma.delete_collection("ncm")
            except Exception:
                pass
            col_ncm = chroma.create_collection("ncm", metadata={"hnsw:space": "cosine"})
            lote = {"ids": [], "emb": [], "doc": [], "meta": []}
            for i, item in enumerate(NCM_LISTA):
                d = item.get("descricao", "")
                # Texto usado na busca: o enriquecido, quando o arquivo trouxer
                busca = item.get("descricao_busca") or d
                if not busca:
                    continue
                lote["ids"].append(f"ncm_{i}"); lote["emb"].append(gerar_embedding(busca))
                lote["doc"].append(busca); lote["meta"].append({"codigo": item.get("codigo",""), "descricao": d})
                if len(lote["ids"]) >= 500:
                    col_ncm.add(ids=lote["ids"], embeddings=lote["emb"], documents=lote["doc"], metadatas=lote["meta"])
                    lote = {"ids": [], "emb": [], "doc": [], "meta": []}
            if lote["ids"]:
                col_ncm.add(ids=lote["ids"], embeddings=lote["emb"], documents=lote["doc"], metadatas=lote["meta"])
            NCM_INDEXADO = True
            globals()["colecao_ncm"] = col_ncm
            print(f"  ✓ NCM indexado: {col_ncm.count()} codigos")

        elif "servicos" in dados or "servico" in nome.lower():
            SERVICO_LISTA = dados.get("servicos", [])
            print(f"\n✓ Codigo de Servico: {len(SERVICO_LISTA)} codigos. Indexando...")
            try:
                chroma.delete_collection("servico")
            except Exception:
                pass
            col_srv = chroma.create_collection("servico", metadata={"hnsw:space": "cosine"})
            for i, item in enumerate(SERVICO_LISTA):
                d = item.get("descricao", "")
                busca = item.get("descricao_busca") or d
                if not busca:
                    continue
                col_srv.add(ids=[f"srv_{i}"], embeddings=[gerar_embedding(busca)],
                            documents=[busca], metadatas=[{"codigo": item.get("codigo",""), "descricao": d}])
            SERVICO_INDEXADO = True
            globals()["colecao_servico"] = col_srv
            print(f"  ✓ Servico indexado: {col_srv.count()} codigos")

    if not uploaded:
        print("Nenhum arquivo. Classificacao seguira sem NCM/codigo de servico.")
except Exception as e:
    print(f"Erro: {e}")


def buscar_ncm(descricao, n=3):
    if not NCM_INDEXADO:
        return []
    res = colecao_ncm.query(query_embeddings=[gerar_embedding(descricao)], n_results=n)
    return [{"codigo": res["metadatas"][0][i]["codigo"], "descricao": res["metadatas"][0][i]["descricao"],
             "score": round(1 - res["distances"][0][i]/2, 3)} for i in range(len(res["ids"][0]))] if res["ids"][0] else []


def buscar_codigo_servico(descricao, n=3):
    if not SERVICO_INDEXADO:
        return []
    res = colecao_servico.query(query_embeddings=[gerar_embedding(descricao)], n_results=n)
    return [{"codigo": res["metadatas"][0][i]["codigo"], "descricao": res["metadatas"][0][i]["descricao"],
             "score": round(1 - res["distances"][0][i]/2, 3)} for i in range(len(res["ids"][0]))] if res["ids"][0] else []

## Bloco 6.5 — Carga inicial de itens corretos (OPCIONAL)

> **Este bloco é opcional.** Pule se quiser começar com a base vazia — muitos sistemas
> estão poluídos com cadastros errados, e nesse caso é melhor construir a base do zero,
> só com itens validados a partir de agora.

Se você tem uma planilha com itens **já classificados corretamente**, este bloco os
indexa no ChromaDB de uma vez, dando ao modelo uma base de referência inicial.

A planilha deve ter as colunas: **descricao**, **grupo**, **subgrupo**, **classe**, **conta**.
Se tiver também **tipo** e **natureza** (produto/servico), elas são aproveitadas.

In [ ]:
from google.colab import files

print("Selecione a planilha de itens JA corretos (ou pule este bloco):")
uploaded = files.upload()

if uploaded:
    nome = list(uploaded.keys())[0]
    base = pd.read_excel(nome) if not nome.endswith(".csv") else pd.read_csv(nome, dtype=str)

    print(f"\n{len(base)} itens. Indexando na base de conhecimento...")
    indexados = 0
    for idx, row in base.iterrows():
        desc = str(row.get("descricao", "")).strip()
        if not desc:
            continue
        try:
            emb = gerar_embedding(desc)
            colecao.upsert(
                ids=[f"inicial_{idx}"],
                embeddings=[emb],
                documents=[desc],
                metadatas=[{
                    "tipo": str(row.get("tipo", "")),
                    "natureza": str(row.get("natureza", "")),
                    "grupo": str(row.get("grupo", "")),
                    "subgrupo": str(row.get("subgrupo", "")),
                    "classe": str(row.get("classe", "")),
                    "conta": str(row.get("conta", "")),
                    "fonte": "carga_inicial"
                }]
            )
            indexados += 1
        except Exception as e:
            print(f"  Erro linha {idx}: {e}")

    print(f"\n✓ {indexados} itens indexados na base de conhecimento")
    print(f"  Total na base agora: {colecao.count()}")
else:
    print("Nenhum arquivo — a base continua vazia. Sem problema!")

## Bloco 6.8 — Anexar orçamento de compra (OPCIONAL)

Se você tem o orçamento do item em **PDF ou imagem**, anexe aqui. O sistema extrai
as informações (valor, especificações, fornecedor) e usa como contexto extra na
classificação — ajudando principalmente na decisão de imobilizar.

> Opcional. Se pular, o bloco 7 funciona normal, só sem o contexto do orçamento.
> O que for extraído fica guardado e é usado automaticamente na próxima classificação.

In [ ]:
from google.colab import files

CONTEXTO_ORCAMENTO = ""

print("Anexe o orcamento (PDF ou imagem). Cancele para pular:")
try:
    uploaded = files.upload()
    if uploaded:
        nome = list(uploaded.keys())[0]
        ext = nome.lower().split(".")[-1]

        if ext == "pdf":
            try:
                import PyPDF2
            except ImportError:
                !pip install -q PyPDF2
                import PyPDF2
            with open(nome, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                texto = "\n".join(page.extract_text() for page in reader.pages)
            CONTEXTO_ORCAMENTO = texto[:2000]
            print(f"\n✓ PDF lido: {len(texto)} caracteres extraidos")

        elif ext in ("png", "jpg", "jpeg", "webp"):
            import base64
            with open(nome, "rb") as f:
                img_b64 = base64.b64encode(f.read()).decode()

            # Modelo de visao atual do Groq (qwen 3.6 27b)
            MODELO_VISAO = "qwen/qwen3.6-27b"
            resp = client.chat.completions.create(
                model=MODELO_VISAO,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Extraia deste documento: descricao do item, valor unitario, quantidade, fornecedor e especificacoes tecnicas. Responda em texto corrido."},
                        {"type": "image_url", "image_url": {"url": f"data:image/{ext};base64,{img_b64}"}}
                    ]
                }],
                max_tokens=500
            )
            CONTEXTO_ORCAMENTO = resp.choices[0].message.content
            print(f"\n✓ Imagem analisada com {MODELO_VISAO}")

        print("\nInformacoes extraidas:")
        print("-"*50)
        print(CONTEXTO_ORCAMENTO[:800])
        print("-"*50)
        print("\nEsse contexto sera usado automaticamente no bloco 7.")
    else:
        print("Nenhum arquivo. O bloco 7 funciona sem contexto de orcamento.")
except Exception as e:
    print(f"Sem orcamento anexado: {e}")
    CONTEXTO_ORCAMENTO = ""

## Bloco 7 — Preparar a classificação

Este bloco **só define as funções** que fazem a classificação. Ele não pede nada e não
classifica nada sozinho, mas precisa ser executado antes dos blocos 7.1, 8 e 10, que usam
essas funções.

**Validação do NCM.** A busca semântica devolve candidatos, não uma decisão. O modelo escolhe
entre os 10 candidatos, a escolha é conferida contra a tabela e, se nada servir, a consulta é
reescrita em termos da nomenclatura e a busca roda de novo (no máximo duas voltas). O resultado
sai com um status: `ok`, `revisar` (confiança baixa ou linha "Outros" havendo linha específica)
ou `nao_encontrado`. Para desligar, mude `VALIDAR_NCM = False` no início do bloco.

**Validação do código de serviço.** Mesma estrutura para a lista da LC 116, com um teste a
mais: o subitem escolhido é cruzado com o subgrupo da taxonomia. Manutenção de máquina na
contabilidade e consultoria na nota não podem coexistir; quando divergem, o item sai como
`revisar`. Para desligar, `VALIDAR_SERVICO = False`.

Se você reiniciar o ambiente do Colab, rode este bloco de novo antes de voltar a classificar.


In [ ]:
REGRAS_IMOB = TAXONOMIA.get("regras_imobilizacao", {})
LIMITE_IMOB = REGRAS_IMOB.get("limite_valor_reais", 1200.0)

import re
import unicodedata


def _norm(texto):
    """Minusculo, sem acento e sem espacos nas pontas."""
    texto = unicodedata.normalize("NFKD", str(texto or "")).encode("ascii", "ignore").decode()
    return texto.lower().strip()


def _sem_numero(texto):
    """Remove a numeracao inicial (ex.: '14. Imobilizados' -> 'imobilizados')."""
    return re.sub(r"^\s*\d+(\.\d+)*\.?\s*", "", _norm(texto))


def normalizar_natureza(valor):
    """Converte a resposta do usuario em 'Produto', 'Servico' ou '' (nao informado)."""
    v = _norm(valor)
    if v in ("p", "prod", "produto", "produtos", "material", "mercadoria", "bem"):
        return "Produto"
    if v in ("s", "serv", "servico", "servicos"):
        return "Servico"
    return ""


def tipos_da_natureza(natureza=""):
    """Tipos compativeis com a natureza. Tipos sem o campo 'natureza' sempre entram."""
    tipos = TAXONOMIA["tipos"]
    if not natureza:
        return tipos
    filtrados = {t: d for t, d in tipos.items()
                 if normalizar_natureza(d.get("natureza", "")) in ("", natureza)}
    return filtrados or tipos


def montar_menu(natureza=""):
    linhas = []
    for tipo, tdata in tipos_da_natureza(natureza).items():
        desc_tipo = tdata.get("descricao", "")
        linhas.append(f"\nTIPO: {tipo}")
        if desc_tipo:
            linhas.append(f"  O QUE ENTRA AQUI: {desc_tipo}")
        if tdata.get("regra"):
            linhas.append(f"  REGRA DO TIPO: {tdata['regra']}")
        for grupo, gdata in tdata.get("grupos", {}).items():
            regra = gdata.get("regra", "")
            extra = f" (usar para: {regra})" if regra else ""
            linhas.append(f"  GRUPO: {grupo}{extra}")
            for sub, sdata in gdata.get("subgrupos", {}).items():
                classes = list(sdata.get("classes", {}).keys())
                regra_sub = f" (usar para: {sdata['regra']})" if sdata.get("regra") else ""
                linhas.append(f"    SUBGRUPO: {sub} [classes: {', '.join(classes)}]{regra_sub}")
    return "\n".join(linhas)


def _destinos_txt(chave, natureza):
    destinos = REGRAS_IMOB.get(chave, {})
    if not isinstance(destinos, dict) or not destinos:
        return ""
    chaves = [natureza] if natureza in destinos else list(destinos.keys())
    return "\n".join(f"  - {k}: {destinos[k]}" for k in chaves)


def montar_regras_imob(natureza=""):
    if not REGRAS_IMOB:
        return "Sem regras de imobilizacao cadastradas."
    crit = "\n".join(f"  - {c}" for c in REGRAS_IMOB.get("criterios", []))
    destino = _destinos_txt("destino_se_imobilizar", natureza) \
        or f"  - {REGRAS_IMOB.get('grupo_destino_se_imobilizar', '')}"
    nao_imob = _destinos_txt("destino_se_nao_imobilizar", natureza)
    nao_imob = f"\nDESTINO SE NAO IMOBILIZAR:\n{nao_imob}" if nao_imob else ""
    return f"""LIMITE PARA IMOBILIZAR: R$ {LIMITE_IMOB:.2f}
VIDA UTIL MINIMA: {REGRAS_IMOB.get('vida_util_minima_meses', 12)} meses
CRITERIOS (atender TODOS para imobilizar):
{crit}
DESTINO SE IMOBILIZAR:
{destino}{nao_imob}"""


def buscar_similares(descricao, n=5):
    if colecao.count() == 0:
        return []
    emb = gerar_embedding(descricao)
    res = colecao.query(query_embeddings=[emb], n_results=min(n, colecao.count()))
    out = []
    if res["ids"] and res["ids"][0]:
        for i, _id in enumerate(res["ids"][0]):
            out.append({
                "id": _id,
                "descricao": res["documents"][0][i],
                "score": round(1 - res["distances"][0][i]/2, 3),
                "meta": res["metadatas"][0][i]
            })
    return out


def _parecido(a, b):
    """Trecho de palavra inteira ou prefixo (evita 'ativo' casar com 'administrativo')."""
    if not a or not b:
        return False
    if a.startswith(b) or b.startswith(a):
        return True
    return bool(re.search(rf"\b{re.escape(a)}\b", b) or re.search(rf"\b{re.escape(b)}\b", a))


def _achar(nome, opcoes):
    """Acha a chave de 'opcoes' que corresponde a 'nome': exata, sem numeracao ou parecida."""
    if not nome or not opcoes:
        return None
    alvo, alvo_sn = _norm(nome), _sem_numero(nome)
    for k in opcoes:
        if _norm(k) == alvo:
            return k
    for k in opcoes:
        if _sem_numero(k) == alvo_sn:
            return k
    for k in opcoes:
        if _parecido(alvo_sn, _sem_numero(k)):
            return k
    return None


CLASSE_ESTOQUE = "Estoque"


def classes_do_subgrupo(tipo, grupo, sub):
    """Classes disponiveis no subgrupo. Procura dentro do TIPO; se nao achar, em todos."""
    tipos = TAXONOMIA["tipos"]
    t = _achar(tipo, tipos)
    for t_nome in ([t] if t else list(tipos.keys())):
        grupos = tipos[t_nome].get("grupos", {})
        g = _achar(grupo, grupos)
        if not g:
            continue
        subs = grupos[g].get("subgrupos", {})
        s = _achar(sub, subs)
        if not s:
            continue
        return subs[s].get("classes", {})
    return {}


def resolver_conta_ref(tipo, grupo, sub, classe):
    """Conta contabil da combinacao tipo > grupo > subgrupo > classe."""
    classes = classes_do_subgrupo(tipo, grupo, sub)
    c = _achar(classe, classes)
    if not c and len(classes) == 1:
        c = next(iter(classes))
    return classes[c].get("conta_ref", "") if c else ""


def localizar(tipo, grupo, sub, classe):
    """
    Troca o que o modelo escreveu pelos nomes EXATOS da taxonomia, com numeracao.
    O modelo costuma responder "Maquinas e Equipamentos" ou colar a descricao do tipo
    junto do nome; o cadastro precisa sempre do mesmo texto, entao a taxonomia manda.
    Devolve tambem a conta da classe encontrada.
    """
    tipos = TAXONOMIA["tipos"]
    t = _achar(tipo, tipos)
    for t_nome in ([t] if t else list(tipos.keys())):
        grupos = tipos[t_nome].get("grupos", {})
        g = _achar(grupo, grupos)
        if not g:
            continue
        subs = grupos[g].get("subgrupos", {})
        s = _achar(sub, subs)
        if not s:
            continue
        classes = subs[s].get("classes", {})
        c = _achar(classe, classes)
        if not c and len(classes) == 1:
            c = next(iter(classes))
        return {"tipo": t_nome, "grupo": g, "subgrupo": s,
                "classe": c or str(classe),
                "conta_ref": classes[c].get("conta_ref", "") if c else "",
                "encontrado": True}
    # Nao achou na taxonomia: devolve como veio, para voce corrigir na validacao
    return {"tipo": str(tipo), "grupo": str(grupo), "subgrupo": str(sub),
            "classe": str(classe), "conta_ref": "", "encontrado": False}


def aplicar_estoque(r, entra_estoque):
    """
    Compra que entra no almoxarifado usa a classe Estoque, quando o subgrupo tem essa
    classe. A conta de resultado nao e decidida aqui: sai na requisicao, pela
    parametrizacao do ERP. Sem estoque, vale a classe do setor que o modelo escolheu.
    """
    classes = classes_do_subgrupo(r.get("tipo", ""), r.get("grupo", ""), r.get("subgrupo", ""))
    disponivel = _achar(CLASSE_ESTOQUE, classes)
    r["estoque"] = bool(entra_estoque and disponivel)
    if r["estoque"]:
        r["classe"] = disponivel
    elif entra_estoque:
        r["aviso_estoque"] = ("Subgrupo sem classe Estoque na taxonomia. "
                              "Mantida a classe do setor.")
    return r


def natureza_do_tipo(tipo):
    """Natureza declarada no tipo da taxonomia; sem declaracao, deduz pelo nome."""
    t = _achar(tipo, TAXONOMIA["tipos"])
    if t:
        n = normalizar_natureza(TAXONOMIA["tipos"][t].get("natureza", ""))
        if n:
            return n
    return "Servico" if "servico" in _norm(tipo) else "Produto"


def mesma_classificacao(a, b):
    """Compara se dois itens tem a MESMA classificacao completa."""
    campos = ["tipo", "grupo", "subgrupo", "classe", "conta"]
    for c in campos:
        va = str(a.get(c, "")).strip().lower()
        vb = str(b.get(c, "")).strip().lower()
        if va != vb:
            return False
    return True


def detectar_versao(descricao, classificacao, similares, limiar=0.75):
    """
    Verifica se ja existe um item com a MESMA classificacao E descricao parecida.
    Se sim, sugere criar nova versao. Retorna (eh_versao, item_pai, proxima_versao).
    """
    for s in similares:
        if s["score"] >= limiar:
            meta = s["meta"]
            if mesma_classificacao(classificacao, meta):
                # Encontrou item com mesma classificacao e descricao parecida
                versao_atual = int(meta.get("versao", "1"))
                item_pai = meta.get("item_pai", s["id"])
                # Descobre a maior versao ja existente desse item pai
                maior = versao_atual
                for outro in similares:
                    if outro["meta"].get("item_pai") == item_pai or outro["id"] == item_pai:
                        maior = max(maior, int(outro["meta"].get("versao", "1")))
                return True, item_pai, maior + 1
    return False, None, 1



# ═══ VALIDACAO DO CODIGO DE SERVICO (LC 116/2003) ═══
# Mesma logica do NCM: o modelo escolhe entre candidatos e a escolha e conferida. A diferenca
# e o cruzamento com a taxonomia: o subgrupo ja diz que servico e esse, entao as duas decisoes
# precisam contar a mesma historia. Se divergirem, o item sai para revisao.

VALIDAR_SERVICO = True
SERVICO_TENTATIVAS = 2
SERVICO_CANDIDATOS = 10
SERVICO_CONFIANCA_MINIMA = 0.6


def escolher_codigo_servico(descricao, aplicacao="", contexto_taxonomia="",
                            n=SERVICO_CANDIDATOS, tentativas=SERVICO_TENTATIVAS):
    """
    Devolve {"codigo", "status", "motivo", "candidatos", "consultas"}.
    status: ok | revisar | nao_encontrado | sem_indice
    """
    if not globals().get("SERVICO_INDEXADO"):
        return {"codigo": "", "status": "sem_indice", "motivo": "Lista da LC 116 nao indexada (bloco 6.2).",
                "candidatos": [], "consultas": []}

    consulta = f"{descricao}. {aplicacao}".strip(". ")
    consultas, cands = [], []
    for tentativa in range(1, max(1, tentativas) + 1):
        consultas.append(consulta)
        cands = buscar_codigo_servico(consulta, n=n)
        if not cands:
            break
        lista = "\n".join(f'{c["codigo"]} | {c["descricao"]}' for c in cands)
        prompt = f"""Voce enquadra servicos na Lista de Servicos da LC 116/2003.

SERVICO: {descricao}
APLICACAO: {aplicacao or "nao informada"}
CLASSIFICACAO CONTABIL JA DEFINIDA: {contexto_taxonomia or "nao informada"}
TENTATIVA: {tentativa} de {tentativas}

CANDIDATOS (unicos valores aceitos):
{lista}

REGRAS:
- Enquadre pela atividade efetivamente prestada, nao pelo nome do fornecedor nem pelo titulo da nota.
- O subitem escolhido deve conversar com a classificacao contabil acima. Se as duas apontarem
  para atividades diferentes, responda coerente = false e explique.
- Se NENHUM candidato servir, devolve codigo vazio e escreve em consulta_sugerida a atividade
  descrita nos termos da lista de servicos.

Responda APENAS JSON:
{{"codigo": "00.00 ou vazio", "confianca": 0.0, "coerente": true, "motivo": "...", "consulta_sugerida": "..."}}"""

        try:
            resp = client.chat.completions.create(
                model=MODELO, messages=[{"role": "user", "content": prompt}],
                temperature=0.1, response_format={"type": "json_object"})
            r = json.loads(resp.choices[0].message.content)
        except Exception as e:
            return {"codigo": cands[0]["codigo"], "status": "revisar",
                    "motivo": f"Falha na validacao ({e}). Mantido o 1o da busca.",
                    "candidatos": cands, "consultas": consultas}

        escolha = str(r.get("codigo", "")).strip()
        validos = {c["codigo"] for c in cands}
        if escolha in validos:
            conf = float(r.get("confianca", 0) or 0)
            motivo = str(r.get("motivo", ""))[:200]
            status = "ok"
            if r.get("coerente") is False:
                status = "revisar"
                motivo = f"Nao bate com a classificacao contabil. {motivo}"
            elif conf < SERVICO_CONFIANCA_MINIMA:
                status = "revisar"
                motivo = f"Confianca baixa ({conf:.0%}). {motivo}"
            return {"codigo": escolha, "status": status, "motivo": motivo,
                    "candidatos": cands, "consultas": consultas}

        nova = str(r.get("consulta_sugerida", "")).strip()
        if not nova or _norm(nova) == _norm(consulta):
            break
        consulta = nova

    return {"codigo": "", "status": "nao_encontrado",
            "motivo": "Nenhum subitem compativel. Enquadre manualmente na lista da LC 116.",
            "candidatos": cands, "consultas": consultas}


# ═══ VALIDACAO DO NCM ═══
# A busca por similaridade devolve candidatos, nao uma decisao. Aqui o modelo escolhe entre
# eles, a escolha e conferida contra a tabela e, se nada servir, a consulta e reescrita em
# termos da nomenclatura e a busca roda de novo. O loop so repete com informacao nova.

VALIDAR_NCM = True          # desligue para usar o primeiro resultado da busca, como antes
NCM_TENTATIVAS = 2          # quantas reescritas de consulta no maximo
NCM_CANDIDATOS = 10         # candidatos enviados ao modelo em cada tentativa
NCM_CONFIANCA_MINIMA = 0.6  # abaixo disso o item sai marcado para revisao


def _ncm_generico(codigo):
    """Termina em 90/99: e a linha 'Outros' da subposicao."""
    return re.sub(r"\D", "", str(codigo))[-2:] in ("90", "99")


def escolher_ncm(descricao, aplicacao="", n=NCM_CANDIDATOS, tentativas=NCM_TENTATIVAS):
    """
    Devolve {"ncm", "status", "motivo", "candidatos", "consultas"}.
    status: ok | revisar | nao_encontrado | sem_indice
    """
    if not globals().get("NCM_INDEXADO"):
        return {"ncm": "", "status": "sem_indice", "motivo": "Tabela NCM nao indexada (bloco 6.2).",
                "candidatos": [], "consultas": []}

    consulta = f"{descricao}. {aplicacao}".strip(". ")
    consultas = []
    for tentativa in range(1, max(1, tentativas) + 1):
        consultas.append(consulta)
        cands = buscar_ncm(consulta, n=n)
        if not cands:
            break
        lista = "\n".join(f'{c["codigo"]} | {c["descricao"]}' for c in cands)
        prompt = f"""Voce classifica mercadorias na NCM/SH brasileira.

ITEM: {descricao}
APLICACAO: {aplicacao or "nao informada"}
TENTATIVA: {tentativa} de {tentativas}

CANDIDATOS (unicos valores aceitos):
{lista}

REGRAS:
- Escolha pelo que o item E (materia, funcao, forma), nao pelo nome comercial.
- So use uma linha "Outros" se nenhuma linha especifica servir.
- Se NENHUM candidato servir, devolve codigo vazio e escreve em consulta_sugerida uma
  descricao do item em termos da nomenclatura, para uma nova busca.

Responda APENAS JSON:
{{"codigo": "0000.00.00 ou vazio", "confianca": 0.0, "motivo": "...", "consulta_sugerida": "..."}}"""

        try:
            resp = client.chat.completions.create(
                model=MODELO, messages=[{"role": "user", "content": prompt}],
                temperature=0.1, response_format={"type": "json_object"})
            r = json.loads(resp.choices[0].message.content)
        except Exception as e:
            return {"ncm": cands[0]["codigo"], "status": "revisar",
                    "motivo": f"Falha na validacao ({e}). Mantido o 1o da busca.",
                    "candidatos": cands, "consultas": consultas}

        escolha = str(r.get("codigo", "")).strip()
        validos = {c["codigo"]: c for c in cands}

        if escolha in validos:
            conf = float(r.get("confianca", 0) or 0)
            motivo = str(r.get("motivo", ""))[:200]
            status = "ok"
            if conf < NCM_CONFIANCA_MINIMA:
                status, motivo = "revisar", f"Confianca baixa ({conf:.0%}). {motivo}"
            elif _ncm_generico(escolha) and any(
                    not _ncm_generico(c["codigo"]) and c["codigo"][:5] == escolha[:5] for c in cands):
                status = "revisar"
                motivo = f"Caiu em linha 'Outros' havendo linha especifica na mesma subposicao. {motivo}"
            return {"ncm": escolha, "status": status, "motivo": motivo,
                    "candidatos": cands, "consultas": consultas}

        nova = str(r.get("consulta_sugerida", "")).strip()
        if not nova or _norm(nova) == _norm(consulta):
            break
        consulta = nova   # informacao nova: a proxima volta busca outra coisa

    return {"ncm": "", "status": "nao_encontrado",
            "motivo": "Nenhum candidato compativel. Classifique o NCM manualmente.",
            "candidatos": cands if "cands" in dir() else [], "consultas": consultas}


def classificar(descricao, aplicacao="", setor="Administrativo", valor=0.0, contexto_orcamento="", natureza="", estoque=False):
    natureza = normalizar_natureza(natureza)
    menu = montar_menu(natureza)
    regras = montar_regras_imob(natureza)
    similares = buscar_similares(descricao)
    sim_txt = "\n".join(
        f"- {s['score']:.0%} | {s['descricao'][:55]} | {s['meta'].get('tipo','?')} > {s['meta'].get('grupo','?')}"
        for s in similares[:3]
    ) or "Nenhum item similar ainda."

    orcamento_txt = f"\nORCAMENTO ANEXADO:\n{contexto_orcamento}" if contexto_orcamento else ""

    if natureza:
        natureza_txt = "PRODUTO (bem material)" if natureza == "Produto" else "SERVICO (prestado por terceiros)"
        regra_natureza = (f"- A NATUREZA foi informada pelo usuario: {natureza_txt}. "
                          "Escolha o TIPO somente entre os listados na taxonomia acima.")
    else:
        natureza_txt = "nao informada"
        regra_natureza = "- A NATUREZA nao foi informada: deduza pela descricao se e PRODUTO ou SERVICO."

    if estoque:
        estoque_txt = "SIM (entra no almoxarifado)"
        regra_estoque = ('- O usuario informou que a compra ENTRA EM ESTOQUE: escolha o subgrupo normalmente e '
                         'use a classe "Estoque" quando o subgrupo tiver essa classe.')
    else:
        estoque_txt = "NAO (consumo imediato)"
        regra_estoque = '- A compra NAO entra em estoque: nao use a classe "Estoque".'

    prompt = f"""Voce classifica itens para cadastro contabil seguindo normas brasileiras (CPC 27 / NBC TG 27).
A taxonomia tem 4 niveis: TIPO > GRUPO > SUBGRUPO > CLASSE.

TAXONOMIA:
{menu}

REGRAS DE IMOBILIZACAO:
{regras}

DECISAO IMOBILIZAR x DESPESA:
{regra_natureza}
- PRODUTO: se atende TODOS os criterios, use o destino de imobilizacao de produto (classe "Ativo").
  Se nao atende, use o tipo de uso e consumo (ou outro tipo de produto da empresa), na classe do setor.
- SERVICO: se agrega valor a um ativo em formacao (obra, montagem, instalacao) e atende os criterios,
  use o grupo de imobilizados dentro de servicos (classe "Ativo"). Caso contrario, e despesa/custo.
- Itens de consumo e pecas de reposicao comuns sao SEMPRE despesa, independente do valor.
- Em despesa/custo, a CLASSE deve corresponder ao setor informado (Producao, Administrativo ou Vendas).
{regra_estoque}
  Na classe "Estoque" o lancamento vai para a conta de estoque; a conta de resultado e definida
  depois, na requisicao, pela parametrizacao do ERP, e nao faz parte desta classificacao.

ITENS SIMILARES:
{sim_txt}

ITEM:
- Natureza: {natureza_txt}
- Descricao: {descricao}
- Aplicacao: {aplicacao}
- Setor: {setor}
- Entra em estoque: {estoque_txt}
- Valor unitario: R$ {valor:.2f}{orcamento_txt}

COMO RESPONDER:
- Copie tipo, grupo, subgrupo e classe EXATAMENTE como aparecem na taxonomia acima, com a
  numeracao na frente (por exemplo "1. Maquinas e Equipamentos", nao "Maquinas e Equipamentos").
- Nao inclua a descricao do tipo nem o texto de "O QUE ENTRA AQUI" dentro do nome.

Retorne APENAS um JSON:
{{"tipo": "...", "grupo": "...", "subgrupo": "...", "classe": "...", "imobilizar": true/false, "confianca": 0.0, "justificativa": "..."}}"""

    resp = client.chat.completions.create(
        model=MODELO,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        response_format={"type": "json_object"}
    )
    r = json.loads(resp.choices[0].message.content)

    # Nomes exatos da taxonomia, nao os que o modelo digitou
    achado = localizar(r.get("tipo", ""), r.get("grupo", ""), r.get("subgrupo", ""), r.get("classe", ""))
    r["tipo"], r["grupo"] = achado["tipo"], achado["grupo"]
    r["subgrupo"], r["classe"] = achado["subgrupo"], achado["classe"]
    if not achado["encontrado"]:
        r["aviso_taxonomia"] = "Combinacao nao encontrada na taxonomia. Ajuste na validacao."

    r = aplicar_estoque(r, estoque)
    ref = resolver_conta_ref(r.get("tipo",""), r.get("grupo",""), r.get("subgrupo",""), r.get("classe",""))
    r["conta_ref"] = ref
    r["conta"] = ref
    r["conta_descricao"] = resolver_conta(ref, INDICE_PLANO) if ref else ""
    r["valor"] = valor
    r["aplicacao"] = aplicacao

    # Natureza: a informada pelo usuario manda; sem ela, vale a do tipo escolhido
    r["natureza"] = natureza or natureza_do_tipo(r.get("tipo", ""))

    # Codigo fiscal: NCM para produtos, Codigo de Servico (LC 116) para servicos
    eh_servico = r["natureza"] == "Servico"
    if eh_servico:
        r["ncm"] = ""
        r["ncm_candidatos"] = []
        contexto_tax = f'{r.get("grupo","")} > {r.get("subgrupo","")}'.strip(" >")
        if globals().get("SERVICO_INDEXADO") and globals().get("VALIDAR_SERVICO", True):
            v = escolher_codigo_servico(descricao, aplicacao, contexto_tax)
            r["codigo_servico"] = v["codigo"]
            r["codigo_servico_status"] = v["status"]
            r["codigo_servico_motivo"] = v["motivo"]
            r["codigo_servico_candidatos"] = v["candidatos"][:3]
            r["codigo_servico_consultas"] = v["consultas"]
        elif globals().get("SERVICO_INDEXADO"):
            srvs = buscar_codigo_servico(descricao, n=3)
            r["codigo_servico"] = srvs[0]["codigo"] if srvs else ""
            r["codigo_servico_status"] = "sem_validacao"
            r["codigo_servico_candidatos"] = srvs
        else:
            r["codigo_servico"] = ""
            r["codigo_servico_status"] = "sem_indice"
            r["codigo_servico_candidatos"] = []
    else:
        r["codigo_servico"] = ""
        r["codigo_servico_candidatos"] = []
        if globals().get("NCM_INDEXADO") and globals().get("VALIDAR_NCM", True):
            v = escolher_ncm(descricao, aplicacao)
            r["ncm"] = v["ncm"]
            r["ncm_status"] = v["status"]
            r["ncm_motivo"] = v["motivo"]
            r["ncm_candidatos"] = v["candidatos"][:3]
            r["ncm_consultas"] = v["consultas"]
        elif globals().get("NCM_INDEXADO"):
            ncms = buscar_ncm(descricao, n=3)
            r["ncm"] = ncms[0]["codigo"] if ncms else ""
            r["ncm_status"] = "sem_validacao"
            r["ncm_candidatos"] = ncms
        else:
            r["ncm"] = ""
            r["ncm_status"] = "sem_indice"
            r["ncm_candidatos"] = []

    r["similares"] = similares
    return r


def subir_para_base(descricao, dados, versao=1, item_pai=None):
    novo_id = f"item_{colecao.count()}_{abs(hash(descricao)) % 10000}"
    emb = gerar_embedding(descricao)
    colecao.upsert(
        ids=[novo_id],
        embeddings=[emb],
        documents=[descricao],
        metadatas=[{
            "natureza": dados.get("natureza", ""),
            "tipo": dados.get("tipo", ""),
            "grupo": dados.get("grupo", ""),
            "subgrupo": dados.get("subgrupo", ""),
            "classe": dados.get("classe", ""),
            "conta": dados.get("conta_ref", ""),
            "estoque": str(dados.get("estoque", False)),
            "ncm": dados.get("ncm", ""),
            "codigo_servico": dados.get("codigo_servico", ""),
            "imobilizar": str(dados.get("imobilizar", False)),
            "aplicacao": dados.get("aplicacao", ""),
            "versao": str(versao),
            "item_pai": item_pai or novo_id,
            "fonte": "validado_humano"
        }]
    )
    return novo_id

print("✓ Funcoes de classificacao carregadas. O bloco 7.1 classifica um item; o bloco 8 faz em lote.")


## Bloco 7.1 — Classificar e validar um item

Primeiro informe se o item é **produto** (P) ou **serviço** (S). Com isso o modelo só considera
os tipos da taxonomia daquela natureza e o sistema já sabe se deve sugerir **NCM** (produto) ou
**Código de Serviço da LC 116** (serviço).

Depois digite a descrição, a **aplicação** (para que será usado), o **setor**, se a compra
**entra em estoque** e o **valor unitário**.

A pergunta de estoque só aparece para produto e é você quem decide, não o modelo. Se a compra
entra no almoxarifado, a classe vira **Estoque** e o cadastro aponta apenas a conta de estoque.
A conta de resultado que recebe a baixa é definida depois, na requisição, pela parametrização
do ERP, e isso não faz parte do classificador. Compra de consumo imediato continua indo direto
para a classe do setor.

O valor é decisivo: com base nas **regras de imobilização** da sua taxonomia
(limite fiscal, vida útil), o modelo decide se o item deve ser **imobilizado (Ativo)**
ou lançado como **estoque/despesa/custo** normal. Produto que atende as regras vai para
**1. Imobilizado**; produto que não atende vai para **2. Uso e Consumo**. Serviço que agrega
valor a um ativo em formação (obra, montagem, instalação) vai para **9. Servicos > 14. Imobilizados**.

Depois de ver o resultado, você valida (s/a/n). Só o que você aprova entra na base.


In [ ]:
# ═══ ENTRADA ═══
natureza = ""
while not natureza:
    natureza = normalizar_natureza(input("E produto ou servico? (P = Produto / S = Servico): "))
    if not natureza:
        print("  Responda P para produto ou S para servico.")
descricao = input(f"Descreva o {'produto' if natureza == 'Produto' else 'servico'}: ")
aplicacao = input("Aplicacao (para que sera usado): ")
setor = input("Setor (Producao/Administrativo/Vendas) [Administrativo]: ") or "Administrativo"
entra_estoque = False
if natureza == "Produto":
    entra_estoque = _norm(input("A compra entra em estoque/almoxarifado? (s/N): ")) in ("s", "sim", "y", "yes", "1")
valor_str = input("Valor unitario de compra (R$) [0]: ") or "0"
try:
    valor = float(valor_str.replace(",", "."))
except:
    valor = 0.0

contexto = globals().get("CONTEXTO_ORCAMENTO", "")

print("\nClassificando...\n")
r = classificar(descricao, aplicacao, setor, valor, contexto, natureza, entra_estoque)

# Detecta se e nova versao de item existente
eh_versao, item_pai, prox_versao = detectar_versao(descricao, r, r["similares"])

print("="*55)
print(f"NATUREZA:   {r.get('natureza','?')}")
print(f"TIPO:       {r.get('tipo','?')}")
print(f"GRUPO:      {r.get('grupo','?')}")
print(f"SUBGRUPO:   {r.get('subgrupo','?')}")
print(f"CLASSE:     {r.get('classe','?')}")
print(f"CONTA:      {r.get('conta_ref','-')}" + (f" - {r['conta_descricao']}" if r.get('conta_descricao') else ""))
if r.get("ncm"):
    selo = {"ok": "", "revisar": "  [CONFERIR]", "sem_validacao": "  [sem validacao]"}.get(r.get("ncm_status"), "")
    print(f"NCM:        {r['ncm']}{selo}")
    if r.get("ncm_motivo") and r.get("ncm_status") != "ok":
        print(f"            {r['ncm_motivo'][:90]}")
    if r.get("ncm_candidatos") and len(r["ncm_candidatos"]) > 1:
        print(f"            (alternativas: {', '.join(c['codigo'] for c in r['ncm_candidatos'][1:3])})")
if r.get("codigo_servico"):
    selo_s = {"ok": "", "revisar": "  [CONFERIR]", "sem_validacao": "  [sem validacao]"}.get(r.get("codigo_servico_status"), "")
    print(f"COD.SERVICO:{r['codigo_servico']}{selo_s}")
    if r.get("codigo_servico_motivo") and r.get("codigo_servico_status") != "ok":
        print(f"            {r['codigo_servico_motivo'][:90]}")
    if r.get("codigo_servico_candidatos") and len(r["codigo_servico_candidatos"]) > 1:
        print(f"            (alternativas: {', '.join(c['codigo'] for c in r['codigo_servico_candidatos'][1:3])})")
imob = r.get("imobilizar", False)
if r.get("estoque"):
    destino = "ESTOQUE (almoxarifado)"
else:
    destino = "IMOBILIZAR (Ativo)" if imob else "DESPESA/CUSTO"
print(f"DECISAO:    {destino}  [limite: R$ {LIMITE_IMOB:.2f} | valor: R$ {valor:.2f}]")
if r.get("estoque"):
    print("            A conta de resultado sai na requisicao, pela parametrizacao do ERP.")
if r.get("aviso_estoque"):
    print(f"⚠ {r['aviso_estoque']}")
if r.get("aviso_taxonomia"):
    print(f"⚠ {r['aviso_taxonomia']}")
if r.get("codigo_servico_status") == "nao_encontrado":
    print("⚠ Nenhum subitem da LC 116 compativel. Enquadre manualmente no ajuste (a).")
if r.get("ncm_status") == "nao_encontrado":
    print("⚠ NCM nao encontrado na tabela para esta descricao. Informe manualmente no ajuste (a).")
print(f"CONFIANCA:  {r.get('confianca',0):.0%}")
print(f"MOTIVO:     {r.get('justificativa','')}")
print("="*55)

if eh_versao:
    print(f"\n⚠ ITEM SIMILAR JA EXISTE com a MESMA classificacao!")
    print(f"  Este item pode ser cadastrado como VERSAO {prox_versao} do item existente,")
    print(f"  em vez de criar um cadastro novo. Isso evita poluir o sistema.")

# ═══ VALIDACAO ═══
print("\nO que deseja fazer?")
print("  s = aprovar como item novo")
if eh_versao:
    print(f"  v = aprovar como VERSAO {prox_versao} do item existente")
print("  a = ajustar antes de subir")
print("  n = descartar")
decisao = input("Escolha: ").strip().lower()

if decisao == "v" and eh_versao:
    versao_desc = input(f"Descricao tecnica que diferencia a versao {prox_versao}: ")
    desc_final = f"{descricao} [v{prox_versao}: {versao_desc}]"
    novo_id = subir_para_base(desc_final, r, versao=prox_versao, item_pai=item_pai)
    print(f"\n✓ Versao {prox_versao} criada (id: {novo_id}, item pai: {item_pai})")
    print(f"  Total na base: {colecao.count()}")
elif decisao == "s":
    novo_id = subir_para_base(descricao, r, versao=1)
    print(f"\n✓ Item novo cadastrado (id: {novo_id}). Total: {colecao.count()}")
elif decisao == "a":
    print("\nAjuste (Enter mantem):")
    r["tipo"] = input(f"  Tipo [{r.get('tipo','')}]: ") or r.get("tipo","")
    r["grupo"] = input(f"  Grupo [{r.get('grupo','')}]: ") or r.get("grupo","")
    r["subgrupo"] = input(f"  Subgrupo [{r.get('subgrupo','')}]: ") or r.get("subgrupo","")
    r["classe"] = input(f"  Classe [{r.get('classe','')}]: ") or r.get("classe","")
    if r.get("ncm"):
        r["ncm"] = input(f"  NCM [{r.get('ncm','')}]: ") or r.get("ncm","")
    if r.get("codigo_servico"):
        r["codigo_servico"] = input(f"  Cod. Servico [{r.get('codigo_servico','')}]: ") or r.get("codigo_servico","")
    achado = localizar(r["tipo"], r["grupo"], r["subgrupo"], r["classe"])
    r["tipo"], r["grupo"] = achado["tipo"], achado["grupo"]
    r["subgrupo"], r["classe"] = achado["subgrupo"], achado["classe"]
    r["conta_ref"] = achado["conta_ref"] or r.get("conta_ref","")
    if not achado["encontrado"]:
        print("  ⚠ Essa combinacao nao existe na taxonomia. Conta nao resolvida.")
    novo_id = subir_para_base(descricao, r, versao=1)
    print(f"\n✓ Ajustado e cadastrado (id: {novo_id}). Total: {colecao.count()}")
else:
    print("\n✗ Descartado.")

## Bloco 8 — Classificar uma planilha inteira (opcional)

Classifica muitos itens de uma vez. O resultado inclui duas colunas em branco —
**aprovado** e **observacao** — para você revisar depois no Excel.

A planilha de entrada deve ter uma coluna **descricao** e, opcionalmente, **natureza**
(produto/servico), **estoque** (s/n), **setor**, **aplicacao** e **valor**. Preencher a natureza
e o estoque melhora a resposta do modelo.

> **Rode o Bloco 7 antes.** Este bloco usa as funções definidas lá; sem isso aparece o erro
> `name 'normalizar_natureza' is not defined`.

> Este bloco **não** sobe nada para a base. A ideia é: classificar em massa aqui,
> revisar no Excel (bloco 9), e subir só os aprovados (bloco 9.5).

In [ ]:
from google.colab import files

print("Selecione a planilha com os itens a classificar:")
uploaded = files.upload()
nome = list(uploaded.keys())[0]
df = pd.read_excel(nome) if not nome.endswith(".csv") else pd.read_csv(nome, dtype=str)

print(f"\n{len(df)} itens. Classificando...\n")
resultados = []
for idx, row in df.iterrows():
    desc = str(row.get("descricao", "")).strip()
    natureza = normalizar_natureza(row.get("natureza", ""))
    entra_estoque = _norm(row.get("estoque", "")) in ("s", "sim", "y", "yes", "1", "true", "x")
    setor = str(row.get("setor", "Administrativo"))
    aplicacao = str(row.get("aplicacao", ""))
    valor_str = str(row.get("valor", "0")).replace(",", ".")
    try:
        valor = float(valor_str)
    except:
        valor = 0.0
    if not desc:
        continue
    try:
        r = classificar(desc, aplicacao, setor, valor, natureza=natureza, estoque=entra_estoque)
        resultados.append({
            "descricao": desc,
            "natureza": r.get("natureza",""),
            "setor": setor,
            "valor": valor,
            "tipo": r.get("tipo",""),
            "grupo": r.get("grupo",""),
            "subgrupo": r.get("subgrupo",""),
            "classe": r.get("classe",""),
            "estoque": r.get("estoque", False),
            "conta": r.get("conta_ref",""),
            "conta_descricao": r.get("conta_descricao",""),
            "ncm": r.get("ncm",""),
            "ncm_status": r.get("ncm_status",""),
            "ncm_motivo": r.get("ncm_motivo",""),
            "codigo_servico": r.get("codigo_servico",""),
            "codigo_servico_status": r.get("codigo_servico_status",""),
            "codigo_servico_motivo": r.get("codigo_servico_motivo",""),
            "imobilizar": r.get("imobilizar", False),
            "confianca": r.get("confianca",0),
            "aprovado": "",
            "observacao": ""
        })
        print(f"  [{idx+1}/{len(df)}] {desc[:38]} -> {r.get('tipo','?')} > {r.get('grupo','?')}")
    except Exception as e:
        print(f"  [{idx+1}] Erro: {e}")

df_result = pd.DataFrame(resultados)
print("\n✓ Classificacao concluida")
print("  Revise a coluna 'aprovado' no Excel (bloco 9) antes de subir a base.")
display(df_result)

## Bloco 9 — Exportar para revisão (opcional)

Salva o resultado do lote em Excel. Abra o arquivo, revise cada linha e preencha
a coluna **aprovado** com **sim** ou **nao**.

- Onde a classificação estiver certa -> escreva `sim`
- Onde estiver errada -> escreva `nao` (ou corrija os campos e escreva `sim`)

Depois, suba o arquivo revisado no bloco 9.5 para alimentar a base só com os aprovados.

In [ ]:
nome_saida = "itens_para_revisao.xlsx"
df_result.to_excel(nome_saida, index=False)
from google.colab import files
files.download(nome_saida)
print(f"✓ {nome_saida} baixado")
print("  Abra, preencha a coluna 'aprovado' com sim/nao, e suba no bloco 9.5")

## Bloco 9.5 — Subir os aprovados para a base (opcional)

Depois de revisar o Excel e preencher a coluna **aprovado**, suba o arquivo aqui.
Este bloco indexa na base de conhecimento **apenas as linhas marcadas como "sim"**.

Mantém o princípio: nada entra na base sem sua aprovação.

In [ ]:
from google.colab import files

print("Selecione a planilha revisada (com a coluna 'aprovado' preenchida):")
uploaded = files.upload()
nome = list(uploaded.keys())[0]
revisado = pd.read_excel(nome) if not nome.endswith(".csv") else pd.read_csv(nome, dtype=str)

if "aprovado" not in revisado.columns:
    print("⚠ A planilha nao tem a coluna 'aprovado'. Revise no bloco 9 primeiro.")
else:
    subidos, ignorados = 0, 0
    for idx, row in revisado.iterrows():
        aprovado = str(row.get("aprovado", "")).strip().lower()
        desc = str(row.get("descricao", "")).strip()
        if aprovado in ("sim", "s", "yes", "1", "true") and desc:
            emb = gerar_embedding(desc)
            novo_id = f"lote_{colecao.count()}_{abs(hash(desc)) % 10000}"
            colecao.upsert(
                ids=[novo_id],
                embeddings=[emb],
                documents=[desc],
                metadatas=[{
                    "natureza": normalizar_natureza(row.get("natureza","")),
                    "tipo": str(row.get("tipo","")),
                    "grupo": str(row.get("grupo","")),
                    "subgrupo": str(row.get("subgrupo","")),
                    "classe": str(row.get("classe","")),
                    "conta": str(row.get("conta","")),
                    "estoque": str(row.get("estoque","")),
                    "ncm": str(row.get("ncm","")),
                    "codigo_servico": str(row.get("codigo_servico","")),
                    "versao": "1",
                    "item_pai": novo_id,
                    "fonte": "lote_aprovado"
                }]
            )
            subidos += 1
        else:
            ignorados += 1
    print(f"\n✓ {subidos} itens aprovados subiram para a base")
    print(f"  {ignorados} ignorados. Total na base: {colecao.count()}")

## Bloco 10 — Medir a precisão (opcional, importante para validação)

Se você tem uma planilha com a classificação **correta** (gabarito), este bloco
compara as sugestões da IA com o gabarito e calcula a precisão.

A planilha deve ter as colunas: **descricao**, **grupo_correto**, **setor**.
Opcionalmente: **natureza** (produto/servico) e **tipo_correto** (para medir também o acerto de tipo).

> **Rode o Bloco 7 antes**, porque este bloco também usa as funções definidas lá.


In [ ]:
from google.colab import files

print("Selecione a planilha de gabarito:")
uploaded = files.upload()
nome = list(uploaded.keys())[0]
gab = pd.read_excel(nome) if not nome.endswith(".csv") else pd.read_csv(nome, dtype=str)

acertos_grupo, acertos_tipo, total, erros = 0, 0, len(gab), []
for idx, row in gab.iterrows():
    desc = str(row["descricao"])
    grupo_correto = str(row.get("grupo_correto", ""))
    tipo_correto = str(row.get("tipo_correto", ""))
    setor = str(row.get("setor", "Administrativo"))
    natureza = normalizar_natureza(row.get("natureza", ""))
    r = classificar(desc, setor=setor, natureza=natureza)

    previsto_grupo = r.get("grupo", "")
    previsto_tipo = r.get("tipo", "")

    # Compara grupo
    grupo_ok = grupo_correto.strip().lower() in previsto_grupo.strip().lower() or previsto_grupo.strip().lower() in grupo_correto.strip().lower()
    if grupo_ok:
        acertos_grupo += 1
    else:
        erros.append({"descricao": desc, "grupo_correto": grupo_correto, "grupo_previsto": previsto_grupo})

    # Compara tipo (se o gabarito tiver a coluna)
    if tipo_correto:
        if tipo_correto.strip().lower() in previsto_tipo.strip().lower() or previsto_tipo.strip().lower() in tipo_correto.strip().lower():
            acertos_tipo += 1

precisao_grupo = acertos_grupo/total*100 if total else 0
print("\n" + "="*55)
print("RESULTADO DA VALIDACAO")
print(f"  Total de itens:       {total}")
print(f"  Acertos de GRUPO:     {acertos_grupo} ({precisao_grupo:.1f}%)")
if any(str(row.get("tipo_correto","")) for _, row in gab.iterrows()):
    precisao_tipo = acertos_tipo/total*100 if total else 0
    print(f"  Acertos de TIPO:      {acertos_tipo} ({precisao_tipo:.1f}%)")
print("="*55)
if erros:
    print("\nErros de grupo:")
    display(pd.DataFrame(erros))

---

## Treinamento do modelo

- **Refinar as regras:** ajuste a taxonomia e as regras para que o modelo fique cada vez mais assertivo.

---

*Classificador Fiscal Inteligente*